In [13]:
import copy
from collections import deque
puzzles = {
    'easy.txt': [
        '530070000','600195000','098000060',
        '800060003','400803001','700020006',
        '060000280','000419005','000080079'
    ],
    'medium.txt': [
        '000260701','680070090','190004500',
        '820100040','004602900','050003028',
        '009300074','040050036','703018000'
    ],
    'hard.txt': [
        '300000000','005009000','200504000',
        '020000700','160000058','704310600',
        '000890100','000067080','000005437'
    ],
    'veryhard.txt': [
        '000000907','000420180','000705026',
        '100904000','050000040','000507009',
        '920108000','034059000','507000000'
    ]
}

step_counter = 0
backtrack_counter = 0
def print_board(board):
    for row in board:
        print(" ".join(str(x) for x in row))
    print()

def parse_board(puzzle):
    return [[int(c) for c in row] for row in puzzle]

def get_neighbors():
    neighbors = {}
    for r in range(9):
        for c in range(9):
            cell = (r, c)
            neighbors[cell] = set()

            for i in range(9):
                neighbors[cell].add((r, i))
                neighbors[cell].add((i, c))

            br, bc = 3 * (r//3), 3 * (c//3)
            for i in range(br, br+3):
                for j in range(bc, bc+3):
                    neighbors[cell].add((i, j))

            neighbors[cell].remove(cell)
    return neighbors

NEIGHBORS = get_neighbors()
def init_domains(board):
    domains = {}
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                domains[(r, c)] = set(range(1, 10))
            else:
                domains[(r, c)] = {board[r][c]}
    return domains
# AC-3
def ac3(domains):
    queue = deque([(xi, xj) for xi in domains for xj in NEIGHBORS[xi]])

    while queue:
        xi, xj = queue.popleft()
        if revise(domains, xi, xj):
            if len(domains[xi]) == 0:
                return False
            for xk in NEIGHBORS[xi]:
                if xk != xj:
                    queue.append((xk, xi))
    return True

def revise(domains, xi, xj):
    revised = False
    for x in set(domains[xi]):
        if not any(x != y for y in domains[xj]):
            domains[xi].remove(x)
            revised = True
    return revised


# FORWARD CHECKING
def forward_check(var, value, domains):
    for neighbor in NEIGHBORS[var]:
        if value in domains[neighbor]:
            domains[neighbor].remove(value)
            if len(domains[neighbor]) == 0:
                return False
    return True


# SELECT VARIABLE (MRV)

def select_unassigned(domains):
    return min(
        (v for v in domains if len(domains[v]) > 1),
        key=lambda var: len(domains[var]),
        default=None
    )


# BACKTRACKING

def backtrack(domains):
    global step_counter, backtrack_counter

    if all(len(domains[v]) == 1 for v in domains):
        return domains

    var = select_unassigned(domains)

    for value in list(domains[var]):
        step_counter += 1

        new_domains = copy.deepcopy(domains)
        new_domains[var] = {value}

        if forward_check(var, value, new_domains):
            if ac3(new_domains):
                result = backtrack(new_domains)
                if result:
                    return result

        backtrack_counter += 1

    return None

# SOLVER

def solve(puzzle_name):
    global step_counter, backtrack_counter

    step_counter = 0
    backtrack_counter = 0

    print(f"\nSolving {puzzle_name}...\n")

    board = parse_board(puzzles[puzzle_name])
    print("Initial Puzzle:")
    print_board(board)

    domains = init_domains(board)

    if not ac3(domains):
        print("No solution found.")
        return

    result = backtrack(domains)

    if result:
        solved = [[list(result[(r,c)])[0] for c in range(9)] for r in range(9)]
        print("Solved Puzzle:")
        print_board(solved)
    else:
        print("No solution found.")

    print(f"Total Steps: {step_counter}")
    print(f"Total Backtracks: {backtrack_counter}")

# RUN ALL
for p in puzzles:
    solve(p)
    print("=====================================")


Solving easy.txt...

Initial Puzzle:
5 3 0 0 7 0 0 0 0
6 0 0 1 9 5 0 0 0
0 9 8 0 0 0 0 6 0
8 0 0 0 6 0 0 0 3
4 0 0 8 0 3 0 0 1
7 0 0 0 2 0 0 0 6
0 6 0 0 0 0 2 8 0
0 0 0 4 1 9 0 0 5
0 0 0 0 8 0 0 7 9

Solved Puzzle:
5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9

Total Steps: 0
Total Backtracks: 0

Solving medium.txt...

Initial Puzzle:
0 0 0 2 6 0 7 0 1
6 8 0 0 7 0 0 9 0
1 9 0 0 0 4 5 0 0
8 2 0 1 0 0 0 4 0
0 0 4 6 0 2 9 0 0
0 5 0 0 0 3 0 2 8
0 0 9 3 0 0 0 7 4
0 4 0 0 5 0 0 3 6
7 0 3 0 1 8 0 0 0

Solved Puzzle:
4 3 5 2 6 9 7 8 1
6 8 2 5 7 1 4 9 3
1 9 7 8 3 4 5 6 2
8 2 6 1 9 5 3 4 7
3 7 4 6 8 2 9 1 5
9 5 1 7 4 3 6 2 8
5 1 9 3 2 6 8 7 4
2 4 8 9 5 7 1 3 6
7 6 3 4 1 8 2 5 9

Total Steps: 0
Total Backtracks: 0

Solving hard.txt...

Initial Puzzle:
3 0 0 0 0 0 0 0 0
0 0 5 0 0 9 0 0 0
2 0 0 5 0 4 0 0 0
0 2 0 0 0 0 7 0 0
1 6 0 0 0 0 0 5 8
7 0 4 3 1 0 6 0 0
0 0 0 8 9 0 1 0 0
0 0 0 